## Rolling updates — a new version with no downtime

The most-used Deployment feature. You change something — almost always the image — and Kubernetes replaces pods a batch at a time, keeping at least `replicas − maxUnavailable` serving the whole time.

```bash
kubectl set image deployment/web app=nginx:1.28-alpine
```

Under the hood when the image changes:

1. The Deployment controller sees `spec.template` has a new hash.
2. It creates a **new ReplicaSet** at `replicas: 0`.
3. It scales the new RS **up by `maxSurge`** while scaling the old **down by `maxUnavailable`**, *in lockstep*. Default 25%/25% on 4 replicas: new→1, old→3, wait for `Ready`, new→2, old→2, and so on.
4. New pods become `Ready` and take traffic (Services route by selector — notebook 04).
5. New RS reaches `replicas`, old reaches `0` → rollout complete.
6. The old RS stays at `0` for rollback, its template preserved exactly.

### The speed-vs-safety knobs

| Setting | Behaviour |
|---|---|
| `maxSurge: 0, maxUnavailable: 1` | Take one down before bringing one up. No spare capacity needed. |
| `maxSurge: 1, maxUnavailable: 0` | One extra up before any old down. Always at full capacity. |
| `25% / 25%` *(default)* | Overlap surge + shutdown — fast, briefly uses extra capacity. |

They can't both be zero (no progress allowed). Watch a rollout with `kubectl rollout status deployment/web` (blocks until done or failed) or `kubectl get pods -l app=web -w`. On our map this is the **Rollout** box lighting **rolling** — the new ReplicaSet climbing as the old one drains.